[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/notebooks/EN/chapter6_seminar_notebook.ipynb)

# Modelling Financial Markets — Seminar 6: Multivariate Volatility and Dependence

*Seminar notebook. Daniel Traian Pele, Antoaneta Amza, Bucharest University of Economic Studies.*

- **[Solved]** problems contain the full solution; **[Proposed]** problems are solved by you.
- Part A: derivations (A1–A6); Part B: estimation and inference (B1–B6); Part C: open question.

> The solutions are discussed in the seminar.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy import stats, optimize, integrate
from scipy.signal import lfilter
from scipy.special import gammaln
try:
    from arch import arch_model
except ImportError:                      # Colab: pip install arch
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)
    from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
FAM_COL = {'gaussian': MainBlue, 't': IDAred, 'clayton': Forest, 'gumbel': Amber, 'frank': Purple}

SEED = 42
NUM = {}
TABLE_DIR = '.'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def shade_crises(ax, alpha=0.12):
    for a, b in (('2008-09-15', '2009-03-31'), ('2020-02-20', '2020-04-30'), ('2022-01-03', '2022-10-31')):
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=Gray, alpha=alpha, lw=0)


import tempfile
TABLE_DIR = tempfile.gettempdir()   # tables are written to a temporary folder


def save_fig(name):
    """Show the figure."""
    plt.show()
    plt.close()

## Data

- Daily prices up to 18 September 2026: SPY, TLT, S&P 500, Euro Stoxx 50, BET, Bitcoin and six banks.
- ETFs and stocks: adjusted closes; indices and Bitcoin: closes.
- Joint analyses: prices are joined on common trading days first, then log returns are computed.
- Weekly returns use the last common close of each week (Friday).

In [ ]:
import os
import numpy as np
import pandas as pd

REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# nume -> (simbol, coloana, eticheta, data de start)
ASSETS = {
    'spy':   ('SPY.US',        'adjusted_close', 'SPY (S&P 500 ETF)',          '2002-07-30'),
    'tlt':   ('TLT.US',        'adjusted_close', 'TLT (20+y Treasury ETF)',    '2002-07-30'),
    'ief':   ('IEF.US',        'adjusted_close', 'IEF (7-10y Treasury ETF)',   '2002-07-30'),
    'sp500': ('GSPC.INDX',     'close',          'S&P 500',                    '2000-01-01'),
    'stoxx': ('STOXX50E.INDX', 'close',          'Euro Stoxx 50',              '2000-01-01'),
    'bet':   ('BET',           'close',          'BET (Romania)',              '2000-01-01'),
    'bettr': ('BETTR.INDX',    'close',          'BET-TR (Romania)',           '2014-09-23'),
    'btc':   ('BTC-USD.CC',    'close',          'Bitcoin',                    '2014-09-17'),
    'vix':   ('VIX.INDX',      'close',          'VIX',                        '2000-01-01'),
    'jpm':   ('JPM.US',        'adjusted_close', 'JPMorgan Chase',             '2010-01-01'),
    'bac':   ('BAC.US',        'adjusted_close', 'Bank of America',            '2010-01-01'),
    'dbk':   ('DBK.XETRA',     'adjusted_close', 'Deutsche Bank',              '2010-01-01'),
    'bnp':   ('BNP.PA',        'adjusted_close', 'BNP Paribas',                '2010-01-01'),
    'tlv':   ('TLV.RO',        'adjusted_close', 'Banca Transilvania',         '2010-01-01'),
    'brd':   ('BRD.RO',        'adjusted_close', 'BRD-Groupe Societe Generale', '2010-01-01'),
}
LABELS = {k: v[2] for k, v in ASSETS.items()}
SHORT = {'spy': 'SPY', 'tlt': 'TLT', 'ief': 'IEF', 'sp500': 'S&P 500', 'stoxx': 'Euro Stoxx 50', 'bet': 'BET',
         'bettr': 'BET-TR', 'btc': 'Bitcoin', 'vix': 'VIX', 'jpm': 'JPM', 'bac': 'BAC', 'dbk': 'DBK', 'bnp': 'BNP',
         'tlv': 'TLV', 'brd': 'BRD'}
BANKS = ['jpm', 'bac', 'dbk', 'bnp', 'tlv', 'brd']


def read_market(symbol):
    """Seria zilnica <SIMBOL> (data/market)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_price(name, start=None, end=END):
    """Pretul zilnic curatat dupa conventiile capitolului."""
    symbol, col, _, start0 = ASSETS[name]
    s = read_market(symbol)[col].loc[start or start0:end]
    s = s[s > 0].dropna()
    if name != 'btc':
        s = s[s.index.dayofweek < 5]          # fara cotatii de weekend
        s = s[s.diff() != 0]                  # fara sarbatori completate cu pretul anterior
    return s.rename(name)


def joint_prices(names, start=None, end=END):
    """Preturile activelor in zilele comune (join interior pe preturi)."""
    return pd.concat([load_price(n, start, end) for n in names], axis=1, join='inner').dropna()


def joint_returns(names, start=None, end=END):
    """Randamente log zilnice calculate DUPA join-ul pe preturi (zile comune)."""
    return np.log(joint_prices(names, start, end)).diff().dropna()


def weekly_returns(names, start=None, end=END):
    """Randamente log saptamanale: ultimul pret comun din fiecare saptamana (vineri), apoi diferente."""
    p = joint_prices(names, start, end)
    return np.log(p.resample('W-FRI').last()).diff().dropna()

## Dependence tools

- EWMA and rolling covariances, Fisher intervals.
- Univariate GARCH(1,1) and the two-step DCC(1,1) of Engle (2002).
- Forbes–Rigobon adjustment, exceedance correlations.
- Five bivariate copulas: density, conditional distribution $h(v\mid u)$, simulation, $\tau$ mapping, tail dependence, canonical maximum likelihood and the Rosenblatt goodness-of-fit test with parametric bootstrap.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats, optimize, integrate
from scipy.signal import lfilter
from scipy.special import gammaln
from arch import arch_model


# =============================================================================
# EWMA si fereastra mobila
# =============================================================================
def ewma_cov(R, lam=0.94, init=100):
    """Sigma_t = lam*Sigma_{t-1} + (1-lam) r_{t-1} r_{t-1}'  (prognoza pentru ziua t, fara r_t)."""
    X = np.asarray(R, dtype=float)
    T, N = X.shape
    S = np.empty((T, N, N))
    S[0] = np.cov(X[:init].T)
    for t in range(1, T):
        S[t] = lam * S[t - 1] + (1 - lam) * np.outer(X[t - 1], X[t - 1])
    return S


def ewma_corr(R, lam=0.94, i=0, j=1):
    S = ewma_cov(R, lam)
    return pd.Series(S[:, i, j] / np.sqrt(S[:, i, i] * S[:, j, j]), index=R.index)


def rolling_corr(R, window, a=None, b=None):
    a, b = a or R.columns[0], b or R.columns[1]
    return R[a].rolling(window).corr(R[b])


def fisher_ci(rho, n, level=0.95):
    """Interval de incredere pentru corelatie prin transformarea Fisher z (date i.i.d.)."""
    z, se = np.arctanh(rho), 1 / np.sqrt(n - 3)
    q = stats.norm.ppf(0.5 + level / 2)
    return np.tanh(z - q * se), np.tanh(z + q * se)


# =============================================================================
# GARCH(1,1) univariat -- primul pas al DCC
# =============================================================================
def garch_filter(r, dist='normal'):
    """GARCH(1,1) cu medie constanta pe randamente in procente (QML).
    Intoarce parametrii, volatilitatea conditionala (%) si reziduurile standardizate."""
    res = arch_model(100 * r, mean='Constant', vol='GARCH', p=1, q=1, dist=dist).fit(disp='off')
    z = (res.resid / res.conditional_volatility).rename(r.name)
    return res.params, res.conditional_volatility.rename(r.name), z, res.loglikelihood


def garch_all(R, dist='normal'):
    out = {c: garch_filter(R[c], dist) for c in R.columns}
    Z = pd.concat([out[c][2] for c in R.columns], axis=1)
    V = pd.concat([out[c][1] for c in R.columns], axis=1)
    P = pd.DataFrame({c: out[c][0] for c in R.columns})
    ll = sum(out[c][3] for c in R.columns)
    return P, V, Z, ll


# =============================================================================
# DCC (Engle, 2002): Q_t = (1-a-b) Qbar + a z_{t-1} z_{t-1}' + b Q_{t-1}
# =============================================================================
def dcc_path(Z, a, b, Qbar=None):
    """Matricele de corelatie R_t (T x N x N). Fiecare element al lui Q_t urmeaza o recursie
    liniara, calculata vectorial cu lfilter."""
    X = np.asarray(Z, dtype=float)
    T, N = X.shape
    Qbar = np.cov(X.T, bias=True) if Qbar is None else Qbar
    P = np.einsum('ti,tj->tij', X, X)                  # z_t z_t'
    drive = np.empty_like(P)
    drive[0] = Qbar
    drive[1:] = (1 - a - b) * Qbar + a * P[:-1]         # intrare la momentul t: foloseste z_{t-1}
    Q = lfilter([1.0], [1.0, -b], drive.reshape(T, -1), axis=0).reshape(T, N, N)
    Q[0] = Qbar
    d = np.sqrt(np.einsum('tii->ti', Q))
    return Q / (d[:, :, None] * d[:, None, :])


def dcc_corr_loglik(Z, a, b, Qbar=None):
    """Partea de corelatie a log-verosimilitatii: -1/2 sum(log|R_t| + z'R_t^{-1}z - z'z)."""
    X = np.asarray(Z, dtype=float)
    R = dcc_path(X, a, b, Qbar)
    sign, logdet = np.linalg.slogdet(R)
    quad = np.einsum('ti,tij,tj->t', X, np.linalg.inv(R), X)
    return -0.5 * np.sum(logdet + quad - np.sum(X ** 2, axis=1))


def ccc_loglik(Z):
    """CCC (Bollerslev, 1990): corelatie constanta = DCC cu a = b = 0."""
    return dcc_corr_loglik(Z, 0.0, 0.0)


def _num_hessian(f, x, h=1e-4):
    k = len(x)
    H = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            e_i, e_j = np.eye(k)[i] * h, np.eye(k)[j] * h
            H[i, j] = (f(x + e_i + e_j) - f(x + e_i - e_j) - f(x - e_i + e_j) + f(x - e_i - e_j)) / (4 * h * h)
    return H


def dcc_fit(Z, start=(0.03, 0.95)):
    """Pasul 2 al DCC: (a, b) prin verosimilitate maxima, cu Qbar fixat la corelatia de selectie
    (correlation targeting). Erorile standard provin din hessiana pasului 2 (ignora eroarea pasului 1)."""
    X = np.asarray(Z, dtype=float)
    Qbar = np.cov(X.T, bias=True)

    def nll(p):
        a, b = p
        if a < 0 or b < 0 or a + b >= 0.9999:
            return 1e10
        return -dcc_corr_loglik(X, a, b, Qbar)

    best = None
    for s in (start, (0.01, 0.98), (0.05, 0.90), (0.10, 0.80)):
        r = optimize.minimize(nll, s, method='Nelder-Mead', options=dict(xatol=1e-7, fatol=1e-7, maxiter=4000))
        if best is None or r.fun < best.fun:
            best = r
    a, b = best.x
    try:
        H = _num_hessian(lambda p: -nll(p), best.x)
        se = np.sqrt(np.diag(np.linalg.inv(-H)))
    except Exception:
        se = np.array([np.nan, np.nan])
    return dict(a=a, b=b, se_a=se[0], se_b=se[1], loglik=-best.fun, Qbar=Qbar,
                R=dcc_path(X, a, b, Qbar), ccc_loglik=ccc_loglik(X))


# =============================================================================
# Forbes-Rigobon (2002)
# =============================================================================
def fr_adjust(rho_c, delta):
    """Corelatia din criza corectata pentru cresterea varianţei pietei-sursa:
    rho* = rho_c / sqrt(1 + delta (1 - rho_c^2)), delta = Var_criza/Var_calm - 1."""
    return rho_c / np.sqrt(1 + delta * (1 - rho_c ** 2))


def fr_inflation(rho, delta):
    """Corelatia masurata in criza cand corelatia adevarata nu se schimba: rho sqrt((1+delta)/(1+delta rho^2))."""
    return rho * np.sqrt((1 + delta) / (1 + delta * rho ** 2))


def fisher_z_test(rho1, n1, rho0, n0):
    """H0: rho1 <= rho0 (test unilateral, esantioane independente)."""
    z = (np.arctanh(rho1) - np.arctanh(rho0)) / np.sqrt(1 / (n1 - 3) + 1 / (n0 - 3))
    return z, 1 - stats.norm.cdf(z)


def crisis_table(ret2, source, targets, calm, crisis):
    """Corelatia calm vs criza, corectia Forbes-Rigobon si testul Fisher z pentru mai multe piete."""
    rows = []
    c0, c1 = ret2.loc[calm[0]:calm[1]], ret2.loc[crisis[0]:crisis[1]]
    delta = c1[source].var() / c0[source].var() - 1
    for tg in targets:
        r0, r1 = c0[source].corr(c0[tg]), c1[source].corr(c1[tg])
        adj = fr_adjust(r1, delta)
        n0, n1 = len(c0) // 2, len(c1) // 2          # randamente pe 2 zile suprapuse: n efectiv ~ n/2
        rows.append(dict(target=tg, rho_calm=r0, rho_crisis=r1, delta=delta, rho_adj=adj,
                         z_raw=fisher_z_test(r1, n1, r0, n0)[0], p_raw=fisher_z_test(r1, n1, r0, n0)[1],
                         z_adj=fisher_z_test(adj, n1, r0, n0)[0], p_adj=fisher_z_test(adj, n1, r0, n0)[1],
                         n_calm=len(c0), n_crisis=len(c1)))
    return pd.DataFrame(rows).set_index('target')


def exceedance_corr(x, y, qs):
    """Corelatia de depasire (Longin-Solnik, 2001): corr(x, y | x<q_x, y<q_y) pentru cozi joase si
    corr(x, y | x>q_x, y>q_y) pentru cozi inalte, la aceleasi cuantile marginale."""
    out = []
    for q in qs:
        lo = (x <= np.quantile(x, q)) & (y <= np.quantile(y, q))
        hi = (x >= np.quantile(x, 1 - q)) & (y >= np.quantile(y, 1 - q))
        out.append((q, np.corrcoef(x[lo], y[lo])[0, 1] if lo.sum() > 5 else np.nan, lo.sum(),
                    np.corrcoef(x[hi], y[hi])[0, 1] if hi.sum() > 5 else np.nan, hi.sum()))
    return pd.DataFrame(out, columns=['q', 'rho_low', 'n_low', 'rho_high', 'n_high']).set_index('q')


# =============================================================================
# Copule bivariate
# =============================================================================
FAMILIES = ['gaussian', 't', 'clayton', 'gumbel', 'frank']
FAM_LABEL = {'gaussian': 'Gaussian', 't': 'Student t', 'clayton': 'Clayton', 'gumbel': 'Gumbel', 'frank': 'Frank'}
EPS = 1e-10


def pseudo_obs(X):
    """Pseudo-observatii: rang / (n + 1), coloana cu coloana."""
    X = np.asarray(X, dtype=float)
    return np.column_stack([stats.rankdata(X[:, k]) / (len(X) + 1) for k in range(X.shape[1])])


def kendall_tau(u, v):
    return stats.kendalltau(u, v)[0]


def spearman_rho(u, v):
    return stats.spearmanr(u, v)[0]


def _debye1(x):
    if abs(x) < 1e-8:
        return 1.0
    val = integrate.quad(lambda t: t / np.expm1(t), 0, abs(x))[0] / abs(x)
    return val if x > 0 else val + abs(x) / 2


def tau_of(fam, theta):
    """Tau Kendall implicat de parametrul copulei."""
    if fam in ('gaussian', 't'):
        return 2 / np.pi * np.arcsin(theta[0] if np.ndim(theta) else theta)
    th = float(np.atleast_1d(theta)[0])
    if fam == 'clayton':
        return th / (th + 2)
    if fam == 'gumbel':
        return 1 - 1 / th
    if fam == 'frank':
        return 1 - 4 / th * (1 - _debye1(th))


def theta_from_tau(fam, tau):
    """Inversarea relatiei tau <-> parametru (metoda momentelor pe tau Kendall)."""
    if fam in ('gaussian', 't'):
        return np.sin(np.pi * tau / 2)
    if fam == 'clayton':
        return 2 * tau / (1 - tau)
    if fam == 'gumbel':
        return 1 / (1 - tau)
    if fam == 'frank':
        return optimize.brentq(lambda th: tau_of('frank', th) - tau, 1e-6 if tau > 0 else -60, 60 if tau > 0 else -1e-6)


def tail_dep(fam, par):
    """(lambda_L, lambda_U) pentru parametrii dati."""
    if fam == 'gaussian':
        return 0.0, 0.0
    if fam == 't':
        rho, nu = par
        lam = 2 * stats.t.cdf(-np.sqrt((nu + 1) * (1 - rho) / (1 + rho)), nu + 1)
        return lam, lam
    th = par[0]
    if fam == 'clayton':
        return 2 ** (-1 / th), 0.0
    if fam == 'gumbel':
        return 0.0, 2 - 2 ** (1 / th)
    return 0.0, 0.0


def log_density(fam, par, u, v):
    u = np.clip(u, EPS, 1 - EPS)
    v = np.clip(v, EPS, 1 - EPS)
    if fam == 'gaussian':
        rho = par[0]
        x, y = stats.norm.ppf(u), stats.norm.ppf(v)
        return -0.5 * np.log(1 - rho ** 2) - (rho ** 2 * (x ** 2 + y ** 2) - 2 * rho * x * y) / (2 * (1 - rho ** 2))
    if fam == 't':
        rho, nu = par
        x, y = stats.t.ppf(u, nu), stats.t.ppf(v, nu)
        return (gammaln((nu + 2) / 2) + gammaln(nu / 2) - 2 * gammaln((nu + 1) / 2) - 0.5 * np.log(1 - rho ** 2)
                - (nu + 2) / 2 * np.log1p((x ** 2 + y ** 2 - 2 * rho * x * y) / (nu * (1 - rho ** 2)))
                + (nu + 1) / 2 * (np.log1p(x ** 2 / nu) + np.log1p(y ** 2 / nu)))
    th = par[0]
    if fam == 'clayton':
        return (np.log1p(th) - (1 + th) * (np.log(u) + np.log(v))
                - (2 + 1 / th) * np.log(u ** -th + v ** -th - 1))
    if fam == 'gumbel':
        lu, lv = -np.log(u), -np.log(v)
        A = lu ** th + lv ** th
        Ath = A ** (1 / th)
        return (-Ath - np.log(u) - np.log(v) + (th - 1) * (np.log(lu) + np.log(lv))
                + (2 / th - 2) * np.log(A) + np.log1p((th - 1) / Ath))
    if fam == 'frank':
        e = -np.expm1(-th)
        den = e - (-np.expm1(-th * u)) * (-np.expm1(-th * v))
        return np.log(th * e) - th * (u + v) - 2 * np.log(np.abs(den))


def h_func(fam, par, u, v):
    """h(v|u) = dC(u,v)/du: functia de repartitie conditionata (transformata Rosenblatt)."""
    u = np.clip(u, EPS, 1 - EPS)
    v = np.clip(v, EPS, 1 - EPS)
    if fam == 'gaussian':
        rho = par[0]
        x, y = stats.norm.ppf(u), stats.norm.ppf(v)
        return stats.norm.cdf((y - rho * x) / np.sqrt(1 - rho ** 2))
    if fam == 't':
        rho, nu = par
        x, y = stats.t.ppf(u, nu), stats.t.ppf(v, nu)
        return stats.t.cdf((y - rho * x) / np.sqrt((nu + x ** 2) * (1 - rho ** 2) / (nu + 1)), nu + 1)
    th = par[0]
    if fam == 'clayton':
        return u ** (-th - 1) * (u ** -th + v ** -th - 1) ** (-1 / th - 1)
    if fam == 'gumbel':
        lu, lv = -np.log(u), -np.log(v)
        A = lu ** th + lv ** th
        return np.exp(-A ** (1 / th)) / u * lu ** (th - 1) * A ** (1 / th - 1)
    if fam == 'frank':
        a, b = np.expm1(-th * u), np.expm1(-th * v)
        return (a + 1) * b / (np.expm1(-th) + a * b)


def simulate(fam, par, n, rng):
    """Simulare de n perechi (u, v) din copula."""
    if fam == 'gaussian':
        rho = par[0]
        z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], n)
        return stats.norm.cdf(z)
    if fam == 't':
        rho, nu = par
        z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], n)
        w = np.sqrt(nu / rng.chisquare(nu, n))
        return stats.t.cdf(z * w[:, None], nu)
    th = par[0]
    if fam == 'clayton':          # Marshall-Olkin cu factor Gamma(1/theta)
        V = rng.gamma(1 / th, 1, n)
        E = rng.exponential(1, (n, 2))
        return (1 + E / V[:, None]) ** (-1 / th)
    if fam == 'gumbel':           # Marshall-Olkin cu factor stabil pozitiv, alpha = 1/theta (Kanter)
        al = 1 / th
        U = rng.uniform(0, np.pi, n)
        W = rng.exponential(1, n)
        S = np.sin(al * U) / np.sin(U) ** (1 / al) * (np.sin((1 - al) * U) / W) ** ((1 - al) / al)
        E = rng.exponential(1, (n, 2))
        return np.exp(-(E / S[:, None]) ** al)
    if fam == 'frank':            # inversarea functiei h
        u, w = rng.uniform(size=n), rng.uniform(size=n)
        v = -np.log1p(w * np.expm1(-th) / (w + (1 - w) * np.exp(-th * u))) / th
        return np.column_stack([u, v])


def fit_copula(fam, u, v):
    """Estimare ML pe pseudo-observatii (pseudo-verosimilitate canonica, CML)."""
    tau = kendall_tau(u, v)
    if fam == 'gaussian':
        f = lambda p: -np.sum(log_density('gaussian', [np.tanh(p[0])], u, v))
        r = optimize.minimize(f, [np.arctanh(np.sin(np.pi * tau / 2))], method='BFGS')
        par = [np.tanh(r.x[0])]
    elif fam == 't':
        f = lambda p: -np.sum(log_density('t', [np.tanh(p[0]), 2.01 + np.exp(p[1])], u, v))
        best = None
        for nu0 in (3, 6, 15):
            r = optimize.minimize(f, [np.arctanh(np.sin(np.pi * tau / 2)), np.log(nu0 - 2.01)], method='Nelder-Mead',
                                  options=dict(xatol=1e-6, fatol=1e-6))
            if best is None or r.fun < best.fun:
                best = r
        r = best
        par = [np.tanh(r.x[0]), 2.01 + np.exp(r.x[1])]
    else:
        lo, hi = {'clayton': (1e-4, 30), 'gumbel': (1.0001, 30), 'frank': (1e-4, 60)}[fam]
        f = lambda p: -np.sum(log_density(fam, [p], u, v))
        th0 = min(max(theta_from_tau(fam, max(tau, 0.02)), lo * 1.01), hi * 0.99)
        r = optimize.minimize_scalar(f, bounds=(lo, hi), method='bounded', options=dict(xatol=1e-7))
        if f(th0) < r.fun:
            r = optimize.minimize_scalar(f, bracket=(lo * 1.01, th0, hi * 0.99), bounds=(lo, hi), method='bounded')
        par = [r.x]
    ll = np.sum(log_density(fam, par, u, v))
    k = 2 if fam == 't' else 1
    lamL, lamU = tail_dep(fam, par)
    return dict(family=fam, par=list(map(float, par)), loglik=float(ll), aic=float(2 * k - 2 * ll),
                bic=float(k * np.log(len(u)) - 2 * ll), tau_model=float(tau_of(fam, par if fam in ('gaussian', 't') else par[0])),
                lamL=float(lamL), lamU=float(lamU))


def cvm_rosenblatt(fam, par, u, v, chunk=1500):
    """S_n^(B) (Genest, Remillard & Beaudoin, 2009): distanta Cramer-von Mises dintre repartitia
    empirica a perechilor Rosenblatt (u, h(v|u)) si copula de independenta."""
    E = np.column_stack([u, h_func(fam, par, u, v)])
    n = len(E)
    t1 = n / 9
    t2 = np.sum((1 - E[:, 0] ** 2) * (1 - E[:, 1] ** 2)) / 2
    t3 = 0.0
    for i in range(0, n, chunk):
        Ei = E[i:i + chunk]
        t3 += np.sum((1 - np.maximum(Ei[:, None, 0], E[None, :, 0])) * (1 - np.maximum(Ei[:, None, 1], E[None, :, 1])))
    return t1 - t2 + t3 / n


def gof_test(fam, u, v, B=200, seed=42, fit=None):
    """Test de adecvare cu bootstrap parametric: simulare din copula estimata, pseudo-observatii,
    re-estimare, statistica S_n^(B); p-valoarea = proportia statisticilor bootstrap >= cea observata."""
    fit = fit or fit_copula(fam, u, v)
    s0 = cvm_rosenblatt(fam, fit['par'], u, v)
    rng = np.random.default_rng(seed)
    sb = []
    for _ in range(B):
        X = simulate(fam, fit['par'], len(u), rng)
        U = pseudo_obs(X)
        fb = fit_copula(fam, U[:, 0], U[:, 1])
        sb.append(cvm_rosenblatt(fam, fb['par'], U[:, 0], U[:, 1]))
    sb = np.array(sb)
    return dict(stat=float(s0), p=float((1 + np.sum(sb >= s0)) / (B + 1)), B=B)


def empirical_tail_dep(u, v, q):
    """lambda_L(q) = P(U<=q, V<=q)/q si lambda_U(q) = P(U>1-q, V>1-q)/q."""
    return np.mean((u <= q) & (v <= q)) / q, np.mean((u > 1 - q) & (v > 1 - q)) / q

## Seminar helpers

- 2-day returns and the crisis windows used in B4.

In [ ]:
CALM08, CRISIS08 = ('2007-09-14', '2008-09-12'), ('2008-09-15', '2009-03-31')
CALM20, CRISIS20 = ('2019-02-19', '2020-02-19'), ('2020-02-20', '2020-04-30')


def two_day_returns(names, start='2006-01-01', end='2021-12-31'):
    """Randamente pe 2 zile (medie mobila, ca in Forbes-Rigobon): log p_t - log p_{t-2}, din preturi comune."""
    lp = np.log(joint_prices(names, start=start, end=end))
    return (lp - lp.shift(2)).dropna()


def bank_copula_data():
    R = joint_returns(['jpm', 'bac'])
    P, V, Z, ll = garch_all(R)
    U = pseudo_obs(Z.values)
    return R, Z, U


def stationary_bootstrap_idx(n, mean_block, rng):
    """Indicii unui bootstrap stationar (Politis-Romano): blocuri de lungime geometrica."""
    idx = np.empty(n, dtype=int)
    idx[0] = rng.integers(n)
    for t in range(1, n):
        idx[t] = rng.integers(n) if rng.uniform() < 1 / mean_block else (idx[t - 1] + 1) % n
    return idx


def simulate_dcc2(a, b, Qbar, eta):
    """Simuleaza reziduuri standardizate bivariate dintr-un DCC(1,1) cu inovatii decorelate eta (T x 2)."""
    T = len(eta)
    eps = np.empty_like(eta)
    q11, q22, q12 = Qbar[0, 0], Qbar[1, 1], Qbar[0, 1]
    c = 1 - a - b
    for t in range(T):
        r = q12 / np.sqrt(q11 * q22)
        eps[t, 0] = eta[t, 0]
        eps[t, 1] = r * eta[t, 0] + np.sqrt(1 - r * r) * eta[t, 1]
        q11 = c * Qbar[0, 0] + a * eps[t, 0] ** 2 + b * q11
        q22 = c * Qbar[1, 1] + a * eps[t, 1] ** 2 + b * q22
        q12 = c * Qbar[0, 1] + a * eps[t, 0] * eps[t, 1] + b * q12
    return eps


def bartlett_lrv(M, L):
    """Varianta pe termen lung (Newey-West, nucleu Bartlett cu L decalaje) a coloanelor lui M (centrate)."""
    M = np.asarray(M, dtype=float)
    M = M - M.mean(0)
    T = len(M)
    S = M.T @ M / T
    for l in range(1, L + 1):
        G = M[l:].T @ M[:-l] / T
        S += (1 - l / (L + 1)) * (G + G.T)
    return S


def corr_grad(m):
    """Gradientul lui rho = (m5 - m1 m2) / sqrt((m3 - m1^2)(m4 - m2^2)), m = (Ex, Ey, Ex^2, Ey^2, Exy)."""
    m1, m2, m3, m4, m5 = m
    vx, vy, c = m3 - m1 ** 2, m4 - m2 ** 2, m5 - m1 * m2
    r = c / np.sqrt(vx * vy)
    return r, np.array([-m2 / np.sqrt(vx * vy) + r * m1 / vx, -m1 / np.sqrt(vx * vy) + r * m2 / vy,
                        -r / (2 * vx), -r / (2 * vy), 1 / np.sqrt(vx * vy)])


def simulate_garch(mu, omega, alpha, beta, eps):
    """Randamente (in %) dintr-un GARCH(1,1) cu medie constanta si inovatiile standardizate eps."""
    T = len(eps)
    r = np.empty(T)
    s2 = omega / (1 - alpha - beta)
    for t in range(T):
        r[t] = mu + np.sqrt(s2) * eps[t]
        s2 = omega + alpha * (r[t] - mu) ** 2 + beta * s2
    return r


def emp_copula_at_points(u, v, chunk=2000):
    n = len(u)
    out = np.empty(n)
    for i in range(0, n, chunk):
        out[i:i + chunk] = np.mean((u[None, :] <= u[i:i + chunk, None]) & (v[None, :] <= v[i:i + chunk, None]), axis=1)
    return out


def tau_se(u, v):
    """Eroarea standard a lui tau Kendall din proiectia Hoeffding: 4 sd(2 C_n(U, V) - U - V) / sqrt(n)."""
    W = 2 * emp_copula_at_points(u, v) - u - v
    return kendall_tau(u, v), 4 * W.std() / np.sqrt(len(u))


def cml_se(fam, par, u, v, hp=1e-5, hu=1e-6):
    """Erori standard CML: inversa hessienei (margini cunoscute) si sandwich-ul Genest-Ghoudi-Rivest (1995),
    in care scorul primeste termenii W1(U_i) + W2(V_i) datorati rangurilor."""
    par = np.asarray(par, float)
    k, n = len(par), len(u)

    def score(p, uu, vv):
        g = np.empty((len(uu), k))
        for j in range(k):
            e = np.zeros(k)
            e[j] = hp * max(1, abs(p[j]))
            g[:, j] = (log_density(fam, p + e, uu, vv) - log_density(fam, p - e, uu, vv)) / (2 * e[j])
        return g
    S = score(par, u, v)
    A = np.zeros((k, k))
    for j in range(k):
        e = np.zeros(k)
        e[j] = 1e-4 * max(1, abs(par[j]))
        A[:, j] = -(score(par + e, u, v) - score(par - e, u, v)).mean(0) / (2 * e[j])
    A = (A + A.T) / 2
    du = (score(par, u + hu, v) - score(par, u - hu, v)) / (2 * hu)
    dv = (score(par, u, v + hu) - score(par, u, v - hu)) / (2 * hu)

    def suffix(x, D):
        o = np.argsort(x)
        cs = np.cumsum(D[o][::-1], axis=0)[::-1] / n
        out = np.empty_like(D)
        out[o] = cs
        return out
    Ai = np.linalg.inv(A)
    Bg = np.atleast_2d(np.cov((S + suffix(u, du) + suffix(v, dv)).T))
    return np.sqrt(np.diag(Ai) / n), np.sqrt(np.diag(Ai @ Bg @ Ai) / n)


N = {}

## Part A — derivations

### A1. Deriving the tail dependence of the t copula [Solved]

- Derivation on the slides; numerical check: $\rho=0.5$; t copula with $\nu\in\{3,4,10,30\}$: $\lambda=2t_{\nu+1}(-\sqrt{(\nu+1)(1-\rho)/(1+\rho)})$.
- Finite levels: $P(V\le q\mid U\le q)$ for $q=10\%, 5\%, 1\%, 0.1\%$.

In [ ]:
# Solution
def a_tail_gauss_t(rho=0.5, nus=(3, 4, 10, 30)):
    """A1: lambda pentru copula t si probabilitatea conditionata P(V<=q | U<=q) la nivel finit."""
    out = {}
    for nu in nus:
        arg = np.sqrt((nu + 1) * (1 - rho) / (1 + rho))
        out[nu] = dict(arg=arg, lam=2 * stats.t.cdf(-arg, nu + 1))
    qs = np.array([0.10, 0.05, 0.01, 0.001])
    mvn = stats.multivariate_normal([0, 0], [[1, rho], [rho, 1]])
    gauss = np.array([mvn.cdf([stats.norm.ppf(q)] * 2) / q for q in qs])
    mvt = stats.multivariate_t([0, 0], [[1, rho], [rho, 1]], df=4)
    t4 = np.array([mvt.cdf([stats.t.ppf(q, 4)] * 2) / q for q in qs])
    return out, qs, gauss, t4


def fig_tail_gauss_t(rho=0.5):
    qs = np.geomspace(0.001, 0.2, 30)
    mvn = stats.multivariate_normal([0, 0], [[1, rho], [rho, 1]])
    g = [mvn.cdf([stats.norm.ppf(q)] * 2) / q for q in qs]
    fig, ax = plt.subplots(figsize=(6.2, 2.9))
    ax.plot(qs, g, color=MainBlue, lw=1.6, label='Gaussian copula (rho = 0.5)')
    for nu, c in ((4, IDAred), (10, Amber)):
        mvt = stats.multivariate_t([0, 0], [[1, rho], [rho, 1]], df=nu)
        ax.plot(qs, [mvt.cdf([stats.t.ppf(q, nu)] * 2) / q for q in qs], color=c, lw=1.6, label=f't copula (rho = 0.5, nu = {nu})')
        lam = 2 * stats.t.cdf(-np.sqrt((nu + 1) * (1 - rho) / (1 + rho)), nu + 1)
        ax.axhline(lam, color=c, ls=':', lw=0.9)
    ax.set_xscale('log')
    ax.set_xlabel('q (log scale)')
    ax.set_ylabel('P(V <= q | U <= q)')
    ax.set_ylim(0, 0.8)
    legend_outside_bottom(ax, ncol=3)
    save_fig('ch6_sem_tail_gauss_t')


lam, qs, gauss, t4 = a_tail_gauss_t()
print({nu: round(d['lam'], 4) for nu, d in lam.items()})
print(pd.DataFrame({'q': qs, 'Gaussian': gauss, 't4': t4}).round(4))
fig_tail_gauss_t()

### A2. Kendall's tau and copula parameters [Solved]

- S&P 500 and Euro Stoxx 50, weekly returns 2000–2026: invert $\hat\tau$ for four copulas; implied tail dependence.

In [ ]:
# Solution
def a_tau_to_theta():
    """A2: tau Kendall pentru S&P 500 / Euro Stoxx 50 (randamente saptamanale) si parametrii implicati."""
    W = weekly_returns(['sp500', 'stoxx'], start='2000-01-01')
    U = pseudo_obs(W.values)
    tau = kendall_tau(U[:, 0], U[:, 1])
    out = dict(tau=tau, n=len(W), rhoS=spearman_rho(U[:, 0], U[:, 1]), pearson=W.corr().iloc[0, 1])
    for fam in ('gaussian', 'clayton', 'gumbel', 'frank'):
        th = theta_from_tau(fam, tau)
        out[fam] = th
        out[fam + '_lam'] = tail_dep(fam, [th])
    out['gauss_rhoS'] = 6 / np.pi * np.arcsin(out['gaussian'] / 2)
    return out, W, U


a2, W2, U2 = a_tau_to_theta()
a2

In [ ]:
# Solution (d)-(e): standard error of tau and the Spearman gap
tau, se_tau = tau_se(U2[:, 0], U2[:, 1])
print('tau', round(tau, 4), 'se', round(se_tau, 4), 'se Clayton', round(2 / (1 - tau) ** 2 * se_tau, 4))
rs_impl = lambda t: 6 / np.pi * np.arcsin(np.sin(np.pi * t / 2) / 2)
rng = np.random.default_rng(SEED)
bs = []
for _ in range(1999):
    Ub = pseudo_obs(W2.values[rng.integers(0, len(W2), len(W2))])
    bs.append(rs_impl(kendall_tau(Ub[:, 0], Ub[:, 1])) - spearman_rho(Ub[:, 0], Ub[:, 1]))
print('gap', round(rs_impl(tau) - spearman_rho(U2[:, 0], U2[:, 1]), 4), '95% CI', np.percentile(bs, [2.5, 97.5]).round(4))

### A3. The Forbes–Rigobon adjustment, step by step [Solved]

- $\rho_{calm}=0.40$, $\rho_c=0.60$, variance ratio 4: $\delta$, $\rho^*$ and the crisis correlation implied by an unchanged $0.40$.

In [ ]:
# Solution
def a_forbes_rigobon_example(rho_calm=0.40, rho_crisis=0.60, var_ratio=4.0):
    delta = var_ratio - 1
    return dict(delta=delta, adj=fr_adjust(rho_crisis, delta), infl=rho_calm * np.sqrt((1 + delta) / (1 + delta * rho_calm ** 2)))


a_forbes_rigobon_example()

In [ ]:
# Solution (d)-(e): delta method with an estimated delta, 2008
def fr_delta_method(rc, delta, n_c, n_0):
    """Eroarea standard a lui rho* = rho_c / sqrt(1 + delta (1 - rho_c^2)): cu delta fixat si cu delta aleator
    (i.i.d. Normal: Var(rho_c) ~ (1 - rho_c^2)^2 / n_c, Var(delta) ~ 2 (1 + delta)^2 (1/(n_c-1) + 1/(n_0-1)))."""
    k = 1 + delta * (1 - rc ** 2)
    g_r, g_d = (1 + delta) / k ** 1.5, -rc * (1 - rc ** 2) / (2 * k ** 1.5)
    v_r, v_d = (1 - rc ** 2) ** 2 / n_c, 2 * (1 + delta) ** 2 * (1 / (n_c - 1) + 1 / (n_0 - 1))
    return np.sqrt(g_r ** 2 * v_r), np.sqrt(g_r ** 2 * v_r + g_d ** 2 * v_d), g_r, g_d, np.sqrt(v_d)


r2 = two_day_returns(['sp500', 'stoxx', 'bet'])
c0, c1 = r2.loc[CALM08[0]:CALM08[1]], r2.loc[CRISIS08[0]:CRISIS08[1]]
delta = c1['sp500'].var() / c0['sp500'].var() - 1
for tg in ('stoxx', 'bet'):
    rc = c1['sp500'].corr(c1[tg])
    se_fix, se_rand = fr_delta_method(rc, delta, len(c1) // 2, len(c0) // 2)[:2]
    print(tg, 'se delta fixed', round(se_fix, 4), 'se delta random', round(se_rand, 4))

### A4. Deriving the Forbes–Rigobon bias [Proposed]

- From $y=\alpha+\beta x+\varepsilon$ derive $\rho_c=\rho\sqrt{(1+\delta)/(1+\delta\rho^2)}$ and invert it. Model: A3.

In [ ]:
# Your code here


### A5. DCC algebra [Proposed]

- Positive definiteness of $Q_t$, $\E[Q_t]=\bar Q$, half-life $\ln0.5/\ln(a+b)$ with the B1 estimates; (d) corrected DCC: $\E[Q_t]=\bar Q^*$. Model: B1.

In [ ]:
# Your code here


### A6. Attainable correlations [Proposed]

- Lognormal margins $X=e^{\sigma_1Z_1}$, $Y=e^{\sigma_2Z_2}$: derive $\rho_{\max}$ and $\rho_{\min}$ from the Fréchet–Hoeffding bounds; evaluate for $\sigma_1=1$, $\sigma_2=2$. Model: A2.

In [ ]:
# Your code here


## Part B — estimation and inference

### B1. DCC for SPY and TLT [Solved]

- GARCH(1,1) + DCC(1,1); likelihood ratio against CCC; subsamples 2002–2019 and 2020–2026; Wied–Krämer–Dehling test for a change in the correlation of $\varepsilon_t$ at an unknown date; two-step standard errors by bootstrap.
- Interpretation: is $a+b$ stable? Why is $\chi^2$ only indicative? What does a change in $\bar Q$ mean?

In [ ]:
# Solution
def b_dcc_spy_tlt():
    """B1: DCC pe tot esantionul si pe doua subperioade; LR fata de CCC; schimbarea corelatiei medii a lui z."""
    R = joint_returns(['spy', 'tlt'])
    P, V, Z, ll = garch_all(R)
    full = dcc_fit(Z)
    sub = {}
    for lab, sl in (('pre', slice(None, '2019-12-31')), ('post', slice('2020-01-01', None))):
        Rs = R.loc[sl]
        Ps, Vs, Zs, _ = garch_all(Rs)
        sub[lab] = dcc_fit(Zs)
        sub[lab]['n'] = len(Zs)
    z0, z1 = Z.loc[:'2021-12-31'], Z.loc['2022-01-01':]
    r0, r1 = z0.corr().iloc[0, 1], z1.corr().iloc[0, 1]
    zt, _ = fisher_z_test(r1, len(z1), r0, len(z0))
    return dict(full=full, sub=sub, r0=r0, r1=r1, zt=zt, p2=2 * (1 - stats.norm.cdf(abs(zt))),
                lr=2 * (full['loglik'] - full['ccc_loglik']), n=len(Z), P=P)


b1 = b_dcc_spy_tlt()
print('full:', round(b1['full']['a'], 4), round(b1['full']['b'], 4), 'LR =', round(b1['lr'], 1))
for k in ('pre', 'post'):
    print(k, round(b1['sub'][k]['a'], 4), round(b1['sub'][k]['b'], 4), round(b1['sub'][k]['Qbar'][0, 1], 3))
print('corr of z before/after 2022:', round(b1['r0'], 3), round(b1['r1'], 3), 'Fisher z =', round(b1['zt'], 2))

In [ ]:
# Solution (d)-(e): unknown-date break test and two-step standard errors (199 replications, a few minutes)
def wkd_test(x, y):
    """Q_T = max_j (j / sqrt(T)) |rho_j - rho_T| / D, D^2 = varianta pe termen lung a corelatiei prin metoda delta,
    nucleu Bartlett cu latimea floor(T^(1/4)) (Wied, Kramer & Dehling, 2012). Sub H0: sup |punte Brownian|."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    T = len(x)
    M = np.column_stack([x, y, x * x, y * y, x * y])
    r, g = corr_grad(M.mean(0))
    D = np.sqrt(g @ bartlett_lrv(M, int(np.floor(T ** 0.25))) @ g)
    C = np.cumsum(M, axis=0) / np.arange(1, T + 1)[:, None]
    m1, m2, m3, m4, m5 = C.T
    with np.errstate(invalid='ignore', divide='ignore'):
        rj = (m5 - m1 * m2) / np.sqrt((m3 - m1 ** 2) * (m4 - m2 ** 2))
    path = np.arange(1, T + 1) / np.sqrt(T) * np.abs(rj - r) / D
    path[:1] = 0
    j = int(np.nanargmax(path))
    Q = path[j]
    return dict(Q=Q, p=stats.kstwobign.sf(Q), j=j, path=path, D=D, rho=r)


def two_step_bootstrap(B=199):
    """Bootstrap parametric cu reziduuri pentru (a, b): (i) doar pasul 2 (DCC reestimat pe reziduurile simulate),
    (ii) ambii pasi (randamente GARCH simulate, GARCH reestimat, apoi DCC). Aceleasi inovatii in (i) si (ii)."""
    R = joint_returns(['spy', 'tlt'])
    P, V, Z, _ = garch_all(R)
    d = dcc_fit(Z)
    X = Z.values
    L = np.linalg.cholesky(d['R'])
    eta = np.linalg.solve(L, X[:, :, None])[:, :, 0]
    eta = (eta - eta.mean(0)) / eta.std(0)
    rng = np.random.default_rng(SEED)
    one, two = [], []
    for _ in range(B):
        eps = simulate_dcc2(d['a'], d['b'], d['Qbar'], eta[rng.integers(0, len(eta), len(eta))])
        d1 = dcc_fit(pd.DataFrame(eps))
        one.append((d1['a'], d1['b']))
        sim = {c: simulate_garch(P.loc['mu', c], P.loc['omega', c], P.loc['alpha[1]', c], P.loc['beta[1]', c], eps[:, k])
               for k, c in enumerate(R.columns)}
        Rs = pd.DataFrame(sim, index=R.index) / 100
        _, _, Zs, _ = garch_all(Rs)
        d2 = dcc_fit(Zs)
        two.append((d2['a'], d2['b']))
    one, two = np.array(one), np.array(two)
    N.update(ts_a=d['a'], ts_b=d['b'], ts_se_a_hess=d['se_a'], ts_se_b_hess=d['se_b'],
             ts_se_a_one=one[:, 0].std(ddof=1), ts_se_b_one=one[:, 1].std(ddof=1),
             ts_se_a_two=two[:, 0].std(ddof=1), ts_se_b_two=two[:, 1].std(ddof=1),
             ts_se_ab_one=one.sum(1).std(ddof=1), ts_se_ab_two=two.sum(1).std(ddof=1),
             ts_mean_a_two=two[:, 0].mean(), ts_mean_b_two=two[:, 1].mean(), ts_B=B)
    return one, two


R = joint_returns(['spy', 'tlt'])
P, V, Z, _ = garch_all(R)
w = wkd_test(Z['spy'], Z['tlt'])
print('Q =', round(w['Q'], 3), 'p =', w['p'], 'break:', Z.index[w['j']].date())
one, two = two_step_bootstrap()
print('se step 2 only', one.std(0, ddof=1).round(4), 'se both steps', two.std(0, ddof=1).round(4))

### B2. Gaussian or t copula for S&P 500 and Euro Stoxx 50? [Solved]

- Canonical maximum likelihood, AIC, bootstrap likelihood ratio for $\nu=\infty$, Rosenblatt goodness of fit (499 bootstrap replications each), bootstrap intervals for $\nu$ and $\lambda$, sandwich standard errors (about half an hour).
- Interpretation: why not $\chi^2_1$? Does passing the fit test prove the model?

In [ ]:
# Solution
def b_gauss_vs_t(W, U, B=200, B_test=499):
    """B2: Gaussian vs t pentru S&P 500 / Euro Stoxx 50 saptamanal: ML, LR pentru nu, adecvare, IC bootstrap pentru lambda.
    Testele (adecvare si LR) folosesc B_test = 499 replicari, deci p-valoarea minima posibila este 1/500 = 0.002."""
    u, v = U[:, 0], U[:, 1]
    fg, ft = fit_copula('gaussian', u, v), fit_copula('t', u, v)
    gg = gof_test('gaussian', u, v, B=B_test, seed=SEED, fit=fg)
    gt = gof_test('t', u, v, B=B_test, seed=SEED, fit=ft)
    lr = 2 * (ft['loglik'] - fg['loglik'])
    rng = np.random.default_rng(SEED)
    lam_b, nu_b = [], []
    for _ in range(B):
        X = pseudo_obs(simulate('t', ft['par'], len(u), rng))
        fb = fit_copula('t', X[:, 0], X[:, 1])
        lam_b.append(fb['lamL'])
        nu_b.append(fb['par'][1])
    lam_b, nu_b = np.array(lam_b), np.array(nu_b)
    # p-valoarea LR prin bootstrap parametric sub H0 (copula Gaussiana), pentru ca nu = infinit e pe frontiera
    lr_b = []
    for _ in range(B_test):
        X = pseudo_obs(simulate('gaussian', fg['par'], len(u), rng))
        lr_b.append(2 * (fit_copula('t', X[:, 0], X[:, 1])['loglik'] - fit_copula('gaussian', X[:, 0], X[:, 1])['loglik']))
    lr_b = np.array(lr_b)
    return dict(fg=fg, ft=ft, gg=gg, gt=gt, lr=lr, lr_p_boot=(1 + np.sum(lr_b >= lr)) / (len(lr_b) + 1),
                lr_p_chi2half=0.5 * (1 - stats.chi2.cdf(lr, 1)), lam_ci=np.percentile(lam_b, [5, 95]),
                nu_ci=np.percentile(nu_b, [5, 95]), emp=[empirical_tail_dep(u, v, q) for q in (0.05, 0.10)])


b2 = b_gauss_vs_t(W2, U2)
print('Gaussian:', b2['fg'])
print('t:', b2['ft'])
print('LR =', round(b2['lr'], 1), 'bootstrap p =', round(b2['lr_p_boot'], 4))
print('fit p: Gaussian', b2['gg']['p'], 't', b2['gt']['p'])
print('nu CI', b2['nu_ci'].round(2), 'lambda CI', b2['lam_ci'].round(3))
for fam, f in (('gaussian', b2['fg']), ('t', b2['ft'])):
    print(fam, 'se inverse Hessian, sandwich:', [x.round(4) for x in cml_se(fam, f['par'], U2[:, 0], U2[:, 1])])

### B3. Clayton or Gumbel for joint losses of BET and Euro Stoxx 50? [Proposed]

- Daily from 2010, GARCH filter, losses $L=-\varepsilon$; Clayton, Gumbel, Gaussian and t; AIC, tail dependence, bootstrap goodness of fit; empirical tails. Model: B2, A2.
- Interpretation: which corner of the loss copula holds the joint crashes?

In [ ]:
# Your code here


### B4. Forbes–Rigobon for 2008 and 2020 [Proposed]

- S&P 500 as source; Euro Stoxx 50 and BET; raw and adjusted tests; shorter crisis windows. Model: A3.
- Interpretation: which assumption is least credible for a pandemic?

In [ ]:
# Your code here


### B5. Bitcoin and the S&P 500: DCC with bootstrap bands [Proposed]

- Residual bootstrap of the DCC parameters (200 replications, a few minutes); 90% band for $\rho_t$. Model: B1.
- Interpretation: is the post-2020 correlation significantly positive? How would you add the first-step GARCH uncertainty?

In [ ]:
# Your code here


### B6. A dynamic t copula for S&P 500 and Euro Stoxx 50 [Proposed]

- GAS t copula with unit scaling on the pseudo-observations of B2; likelihood ratio against the static t copula. Model: B2.
- Interpretation: does the dependence move? Why is $\chi^2$ not the reference? Is B2 contradicted?

In [ ]:
# Your code here


## Part C — did Romanian and euro-area equities become more integrated after 2020? [Proposed]

- BET and Euro Stoxx 50, weekly, 2010–2019 against 2020–2026: block-bootstrap interval for the change in correlation, Forbes–Rigobon adjustment, Kendall's tau, beta, tail dependence, DCC.

In [ ]:
# Your code here
